# NBA Quant AI — GPU Evolution v2 (Kaggle Edition)

**Adapted from nba_gpu_v2.ipynb for Kaggle P100/2xT4 runtime.**

| Optimization | Value |
|---|---|
| Population | 24-40 (auto-scaled by GPU) |
| CV Folds | 2-3 |
| Models | TabICL-focused + tree diversity |
| Feature cache | `/kaggle/working/` (persists across sessions) |
| State save | Every gen (survives crashes) |
| Eval timeout | 90s |
| Data subsample | Last 6000 games |
| Session limit | 9 hours (more gens than Colab 4h) |

---

## IMPORTANT — TWO-PHASE SETUP

**Kaggle disables internet during GPU sessions.** Follow this order:

1. **Internet ON (CPU mode, no GPU needed)**
   - Run **Cell 1** (install deps + clone repo + pre-download TabICL weights)
   - Run **Cell 2** (build feature cache → saved to `/kaggle/working/`)
   - Takes ~30-40 min. Cache persists for future sessions automatically.

2. **Switch to GPU** (Settings → Accelerator → GPU P100 or T4)
   - Internet will be OFF automatically
   - Run **Cells 3-6** — cache already on disk, no internet needed

**Secrets required** (Add → Secrets in Kaggle sidebar):
- `HF_TOKEN` — HuggingFace token (island seed fetching, only Cell 3)
- `DATABASE_URL` — Supabase pooler connection string

**Target to beat:** 0.21837 (S13 CatBoost gen815)

In [ ]:
# ============================================================
# CELL 1: SETUP — DEPS + REPO CLONE + MODEL PRE-DOWNLOAD
# RUN WITH INTERNET ON (CPU mode) — only needed once per session
# ============================================================
import subprocess, sys, os, time, gc, json, warnings
warnings.filterwarnings('ignore')

CACHE_DIR = '/kaggle/working'
os.makedirs(CACHE_DIR, exist_ok=True)
print(f'Cache dir: {CACHE_DIR}')

# Kaggle secrets (internet must be ON)
try:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient()
    for key in ['HF_TOKEN', 'DATABASE_URL']:
        try:
            v = secret.get_secret(key)
            if v:
                os.environ[key] = v
                print(f'{key}: OK')
        except Exception as e:
            print(f'{key}: not set ({e})')
except ImportError:
    print('kaggle_secrets not available — running outside Kaggle?')

# Install dependencies (requires internet ON)
print('\nInstalling dependencies...')
t0 = time.time()
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'xgboost', 'lightgbm', 'catboost', 'scikit-learn', 'pandas', 'numpy',
    'scipy', 'tabicl', 'huggingface_hub', 'psycopg2-binary', 'requests'])
print(f'Deps installed: {time.time()-t0:.0f}s')

# PyTorch is pre-installed on Kaggle
import torch, numpy as np
print(f'PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        mem_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'GPU {i}: {torch.cuda.get_device_name(i)} ({mem_gb:.1f} GB)')
else:
    print('CPU mode (expected for internet-ON phase)')

# Clone nomos-nba-agent (requires internet ON)
REPO_DIR = '/kaggle/working/nomos-nba-agent'
if not os.path.exists(REPO_DIR):
    print('\nCloning nomos-nba-agent...')
    t0 = time.time()
    subprocess.run(['git', 'clone', '--depth=1',
                    'https://github.com/LBJLincoln/nomos-nba-agent.git',
                    REPO_DIR], check=True)
    print(f'Cloned in {time.time()-t0:.0f}s')
else:
    print(f'Repo already at {REPO_DIR}')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], capture_output=True)

# Pre-download TabICL checkpoint while internet is ON.
# Weights cached to ~/.cache/huggingface — survives GPU switch.
print('\nPre-downloading TabICL checkpoint (requires internet ON)...')
t0 = time.time()
try:
    from tabicl import TabICLClassifier
    _m = TabICLClassifier()
    _m.fit(np.random.randn(50, 5), np.random.randint(0, 2, 50))
    del _m
    gc.collect()
    print(f'TabICL checkpoint cached ({time.time()-t0:.0f}s)')
except Exception as e:
    print(f'TabICL pre-download FAILED: {e}')
    print('WARNING: TabICL will not work in offline GPU phase.')

print('\nCell 1 complete. Proceed to Cell 2 to build feature cache.')


In [ ]:
# ============================================================
# CELL 2: BUILD OR LOAD FEATURES (cached in /kaggle/working/)
# RUN WITH INTERNET ON (CPU mode) — cache persists across sessions
# ============================================================
from pathlib import Path
import sys, os, json, time
import numpy as np

CACHE_DIR = '/kaggle/working'
REPO_DIR = '/kaggle/working/nomos-nba-agent'
CACHE_FILE = Path(f'{CACHE_DIR}/features_cache_v38.npz')

if CACHE_FILE.exists():
    print('Loading cached features from /kaggle/working/...')
    t0 = time.time()
    cached = np.load(str(CACHE_FILE), allow_pickle=True)
    X_full = cached['X']
    y_full = cached['y']
    feature_names = cached['feature_names'].tolist()
    print(f'Loaded: {X_full.shape} in {time.time()-t0:.1f}s')
else:
    print('Building features (~30-40 min, will cache to /kaggle/working/)...')
    t0 = time.time()

    if not os.path.exists(REPO_DIR):
        raise RuntimeError('Repo not found — did Cell 1 complete successfully?')

    sys.path.insert(0, f'{REPO_DIR}/hf-space')

    data_dir = Path(f'{REPO_DIR}/hf-space/data/historical')
    games = []
    for f in sorted(data_dir.glob('games-*.json')):
        raw = json.loads(f.read_text())
        if isinstance(raw, list):
            games.extend(raw)
        elif isinstance(raw, dict) and 'games' in raw:
            games.extend(raw['games'])
    print(f'Games loaded: {len(games)}')

    from features.engine import NBAFeatureEngine
    engine = NBAFeatureEngine()
    X_raw, y_raw, feature_names = engine.build(games)
    X_full = np.nan_to_num(np.array(X_raw, dtype=np.float32))
    y_full = np.array(y_raw, dtype=np.int32)

    variances = np.var(X_full, axis=0)
    valid = variances > 1e-10
    X_full = X_full[:, valid]
    feature_names = [f for f, v in zip(feature_names, valid) if v]

    np.savez_compressed(str(CACHE_FILE), X=X_full, y=y_full,
                        feature_names=np.array(feature_names))
    print(f'Built & cached: {X_full.shape} in {time.time()-t0:.0f}s')

SUBSAMPLE = 6000
if X_full.shape[0] > SUBSAMPLE:
    X = X_full[-SUBSAMPLE:]
    y = y_full[-SUBSAMPLE:]
    print(f'Subsampled: {X_full.shape[0]} -> {SUBSAMPLE} (last {SUBSAMPLE} games)')
else:
    X = X_full
    y = y_full

n_features = X.shape[1]
print(f'Ready: {X.shape} ({n_features} features)')
print('\nCell 2 complete.')
print('NOW: Switch to GPU accelerator (Settings -> Accelerator -> GPU P100 or T4),')
print('then run Cells 3-6. Internet will be OFF — cache handles everything.')


In [ ]:
# ============================================================
# CELL 3: SEED POPULATION FROM 6 ISLANDS + CONFIG
# Run with GPU ON (internet OFF — island fetch fails gracefully)
# ============================================================
import requests, random, signal, gc, json, time, warnings
import numpy as np
import torch
from pathlib import Path
from collections import Counter
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import brier_score_loss
from sklearn.ensemble import ExtraTreesClassifier
import xgboost as xgb
import lightgbm as lgb
warnings.filterwarnings('ignore')

CACHE_DIR = '/kaggle/working'
REPO_DIR = '/kaggle/working/nomos-nba-agent'
CACHE_FILE = Path(f'{CACHE_DIR}/features_cache_v38.npz')

# Reload features if kernel was restarted after Cell 2
if 'X' not in dir() or 'feature_names' not in dir():
    print('Reloading feature cache...')
    import sys
    sys.path.insert(0, f'{REPO_DIR}/hf-space')
    cached = np.load(str(CACHE_FILE), allow_pickle=True)
    X_full = cached['X']
    y_full = cached['y']
    feature_names = cached['feature_names'].tolist()
    SUBSAMPLE = 6000
    if X_full.shape[0] > SUBSAMPLE:
        X = X_full[-SUBSAMPLE:]
        y = y_full[-SUBSAMPLE:]
    else:
        X = X_full
        y = y_full
    n_features = X.shape[1]
    print(f'Reloaded: {X.shape}')

# GPU detection — Kaggle offers P100 (16GB) or 2xT4 (2x16GB)
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        mem_gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'GPU {i}: {torch.cuda.get_device_name(i)} ({mem_gb:.1f} GB)')
    if n_gpu >= 2:
        PLATFORM = 'kaggle_2xT4'
    else:
        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        PLATFORM = 'kaggle_P100' if gpu_mem > 14 else 'kaggle_T4'
else:
    PLATFORM = 'kaggle_cpu'
    print('WARNING: No GPU detected.')

print(f'Platform detected: {PLATFORM}')

# Kaggle: 9h session limit -> more gens than Colab 4h
CONFIGS = {
    'kaggle_P100': {'POP': 32, 'FOLDS': 2, 'GENS': 800, 'ELITE': 5, 'TIMEOUT': 90},
    'kaggle_T4':   {'POP': 24, 'FOLDS': 2, 'GENS': 600, 'ELITE': 4, 'TIMEOUT': 90},
    'kaggle_2xT4': {'POP': 40, 'FOLDS': 3, 'GENS': 800, 'ELITE': 6, 'TIMEOUT': 120},
    'kaggle_cpu':  {'POP': 16, 'FOLDS': 2, 'GENS': 200, 'ELITE': 3, 'TIMEOUT': 120},
}
cfg = CONFIGS.get(PLATFORM, CONFIGS['kaggle_T4'])
POP_SIZE     = cfg['POP']
N_SPLITS     = cfg['FOLDS']
TOTAL_GENS   = cfg['GENS']
ELITE_SIZE   = cfg['ELITE']
EVAL_TIMEOUT = cfg['TIMEOUT']

PURGE_GAP       = 5
TARGET_FEATURES = 60
MAX_FEATURES    = 200
MUTATION_RATE   = 0.10
CROSSOVER_RATE  = 0.80
MUT_FLOOR       = 0.05
MUT_DECAY       = 0.998

_xgb_device = 'cuda' if torch.cuda.is_available() else 'cpu'
GPU_MODEL_TYPES = ['tabicl', 'xgboost', 'catboost', 'lightgbm', 'extra_trees']
MODEL_WEIGHTS   = {'tabicl': 50, 'xgboost': 15, 'catboost': 10, 'lightgbm': 10, 'extra_trees': 15}

tscv = TimeSeriesSplit(n_splits=N_SPLITS)
splits = [(tr[:-PURGE_GAP] if len(tr) > PURGE_GAP + 50 else tr, te)
          for tr, te in tscv.split(X)]

print(f'\nPlatform: {PLATFORM} | Pop={POP_SIZE} | Folds={N_SPLITS} | Gens={TOTAL_GENS}')
print(f'XGBoost device: {_xgb_device}')
print(f'Estimated: ~{POP_SIZE * 8}s/gen = ~{3600 // max(POP_SIZE * 8, 1)} gens/hour')
print(f'Session budget: 9h -> ~{9 * 3600 // max(POP_SIZE * 8, 1)} gens possible')

# Fetch seeds from 6 HF islands (will skip gracefully if internet is OFF)
ISLANDS = {
    'S10': 'https://nomos42-nba-quant.hf.space',
    'S11': 'https://nomos42-nba-quant-2.hf.space',
    'S12': 'https://nomos42-nba-evo-3.hf.space',
    'S13': 'https://nomos42-nba-evo-4.hf.space',
    'S14': 'https://nomos42-nba-evo-5.hf.space',
    'S15': 'https://nomos42-nba-evo-6.hf.space',
}

island_seeds = []
print('\nFetching seeds from islands (requires internet — will skip if offline)...')
for name, url in ISLANDS.items():
    try:
        resp = requests.get(f'{url}/api/results', timeout=10)
        if resp.status_code != 200:
            continue
        data = resp.json()
        best = data.get('best', {})
        island_seeds.append({
            'source': name, 'brier': best.get('brier', 1.0),
            'features': best.get('selected_features', []),
            'model_type': best.get('model_type', 'xgboost'),
        })
        brier_val = best.get('brier', '?')
        mt_val = best.get('model_type', '?')
        nf_val = best.get('n_features', '?')
        print(f'  {name}: brier={brier_val} model={mt_val} feat={nf_val}')
        for i, ind in enumerate(data.get('top5', [])[:2]):
            island_seeds.append({
                'source': f'{name}_top{i+1}', 'brier': ind.get('brier', 1.0),
                'features': ind.get('selected_features', []),
                'model_type': ind.get('model_type', 'xgboost'),
            })
    except Exception as e:
        print(f'  {name}: offline ({type(e).__name__}) — will use random init')

print(f'Seeds collected: {len(island_seeds)}')


In [ ]:
# ============================================================
# CELL 4: EVOLUTION ENGINE
# ============================================================

def make_model(model_type, hp):
    if model_type == 'xgboost':
        return xgb.XGBClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 6),
            learning_rate=hp.get('learning_rate', 0.05),
            subsample=hp.get('subsample', 0.8),
            colsample_bytree=hp.get('colsample_bytree', 0.8),
            tree_method='hist', device=_xgb_device,
            random_state=42, verbosity=0)
    elif model_type == 'lightgbm':
        return lgb.LGBMClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 6),
            learning_rate=hp.get('learning_rate', 0.05),
            subsample=hp.get('subsample', 0.8),
            colsample_bytree=hp.get('colsample_bytree', 0.8),
            random_state=42, verbose=-1)
    elif model_type == 'extra_trees':
        return ExtraTreesClassifier(
            n_estimators=min(hp.get('n_estimators', 200), 300),
            max_depth=hp.get('max_depth', 8),
            min_samples_leaf=5, random_state=42, n_jobs=-1)
    elif model_type == 'catboost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(
            iterations=min(hp.get('n_estimators', 200), 300),
            depth=min(hp.get('max_depth', 6), 10),
            learning_rate=hp.get('learning_rate', 0.05),
            random_state=42, verbose=0)
    elif model_type == 'tabicl':
        from tabicl import TabICLClassifier
        return TabICLClassifier()
    else:
        return ExtraTreesClassifier(n_estimators=200, random_state=42, n_jobs=-1)


class _Timeout(Exception):
    pass

def _timeout_handler(signum, frame):
    raise _Timeout()


def evaluate(features_mask, model_type, hp, timeout=EVAL_TIMEOUT):
    indices = [i for i, v in enumerate(features_mask) if v]
    if len(indices) < 5:
        return 1.0
    if len(indices) > MAX_FEATURES:
        indices = indices[:MAX_FEATURES]
    X_sub = X[:, indices].astype(np.float32)

    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(timeout)

    try:
        fold_briers = []
        for train_idx, test_idx in splits:
            model = make_model(model_type, hp)
            model.fit(X_sub[train_idx], y[train_idx])
            probs = model.predict_proba(X_sub[test_idx])[:, 1]
            fold_briers.append(brier_score_loss(y[test_idx], probs))
            del model
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return float(np.mean(fold_briers))
    except _Timeout:
        signal.signal(signal.SIGALRM, old_handler)
        gc.collect()
        return 0.30
    except Exception:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)
        return 0.30


class Individual:
    def __init__(self, features=None, model_type=None, hp=None):
        if features is None:
            prob = TARGET_FEATURES / max(n_features, 1)
            self.features = [1 if random.random() < prob else 0 for _ in range(n_features)]
        else:
            self.features = list(features)
        self.model_type = model_type or random.choices(
            list(MODEL_WEIGHTS.keys()), weights=list(MODEL_WEIGHTS.values()), k=1)[0]
        self.hp = hp or {
            'n_estimators': random.randint(100, 300),
            'max_depth': random.randint(4, 9),
            'learning_rate': 10 ** random.uniform(-2.0, -0.7),
            'subsample': random.uniform(0.6, 1.0),
            'colsample_bytree': random.uniform(0.4, 1.0),
        }
        self.brier = 1.0
        self.n_feat = sum(self.features)
        self._enforce_cap()

    def _enforce_cap(self):
        indices = [i for i, v in enumerate(self.features) if v]
        if len(indices) > MAX_FEATURES:
            for i in random.sample(indices, len(indices) - MAX_FEATURES):
                self.features[i] = 0
        self.n_feat = sum(self.features)

    def eval(self):
        self.brier = evaluate(self.features, self.model_type, self.hp)
        return self.brier

    def mutate(self, rate):
        for i in range(len(self.features)):
            if random.random() < rate:
                self.features[i] = 1 - self.features[i]
        if random.random() < 0.25:
            self.hp['n_estimators'] = max(50, min(300, self.hp['n_estimators'] + random.randint(-50, 50)))
        if random.random() < 0.25:
            self.hp['max_depth'] = max(2, min(10, self.hp['max_depth'] + random.randint(-2, 2)))
        if random.random() < 0.25:
            self.hp['learning_rate'] = max(0.001, min(0.3,
                self.hp['learning_rate'] * 10 ** random.uniform(-0.3, 0.3)))
        if random.random() < 0.08:
            self.model_type = random.choices(
                list(MODEL_WEIGHTS.keys()), weights=list(MODEL_WEIGHTS.values()), k=1)[0]
        self._enforce_cap()
        self.brier = 1.0

    @staticmethod
    def crossover(p1, p2):
        point = random.randint(1, len(p1.features) - 1)
        child = Individual(
            features=p1.features[:point] + p2.features[point:],
            model_type=p1.model_type if random.random() < 0.5 else p2.model_type,
            hp={k: p1.hp[k] if random.random() < 0.5 else p2.hp[k] for k in p1.hp})
        return child


# Build or resume population
name_to_idx = {name: i for i, name in enumerate(feature_names)}
STATE_FILE = Path(f'{CACHE_DIR}/evolution_state_kaggle.json')
population = []
best_ever_brier = 1.0
best_ever_info = None
start_gen = 0

if STATE_FILE.exists():
    try:
        state = json.loads(STATE_FILE.read_text())
        start_gen = state.get('generation', 0)
        best_ever_brier = state.get('best_brier', 1.0)
        best_ever_info = state.get('best_info')
        for saved in state.get('population', []):
            feat = [0] * n_features
            for fname in saved.get('features', []):
                if fname in name_to_idx:
                    feat[name_to_idx[fname]] = 1
            ind = Individual(features=feat, model_type=saved.get('model_type', 'tabicl'),
                             hp=saved.get('hp', {}))
            ind.brier = saved.get('brier', 1.0)
            population.append(ind)
        print(f'RESUMED: gen={start_gen}, best={best_ever_brier:.5f}, pop={len(population)}')
    except Exception as e:
        print(f'State load failed: {e} — starting fresh')

if len(population) < POP_SIZE:
    for seed in island_seeds:
        if len(population) >= POP_SIZE:
            break
        feat = [0] * n_features
        for fname in seed.get('features', []):
            if isinstance(fname, str) and fname in name_to_idx:
                feat[name_to_idx[fname]] = 1
        if sum(feat) < 15:
            prob = TARGET_FEATURES / max(n_features, 1)
            feat = [1 if random.random() < prob else 0 for _ in range(n_features)]
        population.append(Individual(features=feat, model_type='tabicl'))
        if len(population) < POP_SIZE:
            population.append(Individual(features=list(feat),
                                         model_type=seed.get('model_type', 'xgboost')))

    while len(population) < POP_SIZE:
        population.append(Individual())
    population = population[:POP_SIZE]

mt_counts = Counter(ind.model_type for ind in population)
print(f'Population ready: {len(population)} | Models: {dict(mt_counts)}')


In [ ]:
# ============================================================
# CELL 5: EVOLUTION LOOP (9-hour Kaggle session budget)
# State saved every gen to /kaggle/working/ — survives crashes
# ============================================================
print('='*70)
print(f'  NBA QUANT AI — GPU EVOLUTION v2 (Kaggle Edition)')
print(f'  Platform: {PLATFORM} | Pop={POP_SIZE} | Folds={N_SPLITS} | Gens={TOTAL_GENS}')
print(f'  Resume from gen: {start_gen}')
print(f'  ATR to beat: 0.21837 (S13 CatBoost gen815)')
print(f'  Session budget: 9 hours')
print('='*70)

mut_rate = MUTATION_RATE
session_start = time.time()
gens_this_session = 0
SESSION_LIMIT_HOURS = 8.5  # stop before Kaggle 9h hard kill


def save_state(gen, population, best_brier, best_info, mut_rate):
    # State saved to /kaggle/working/ every generation.
    # /kaggle/working/ is cleared when session ends unless committed.
    # For cross-session persistence: download evolution_state_kaggle.json
    # and re-upload as a Kaggle Dataset, or use Save & Run All.
    pop_data = []
    for ind in population:
        pop_data.append({
            'features': [feature_names[i] for i, v in enumerate(ind.features) if v],
            'model_type': ind.model_type,
            'hp': ind.hp,
            'brier': ind.brier,
        })
    STATE_FILE.write_text(json.dumps({
        'generation': gen + 1,
        'best_brier': best_brier,
        'best_info': best_info,
        'mutation_rate': mut_rate,
        'population': pop_data,
        'platform': PLATFORM,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, default=str))


for gen in range(start_gen, TOTAL_GENS):
    elapsed_h = (time.time() - session_start) / 3600
    if elapsed_h >= SESSION_LIMIT_HOURS:
        print(f'\nSession limit reached ({elapsed_h:.1f}h >= {SESSION_LIMIT_HOURS}h). Stopping.')
        print('State saved. Resume next session by re-running from Cell 3.')
        break

    gen_start = time.time()

    n_eval = 0
    for ind in population:
        if ind.brier >= 0.99:
            ind.eval()
            n_eval += 1

    population.sort(key=lambda x: x.brier)

    gen_best = population[0]
    improved = gen_best.brier < best_ever_brier
    if improved:
        best_ever_brier = gen_best.brier
        best_ever_info = {
            'brier': gen_best.brier, 'model_type': gen_best.model_type,
            'n_features': gen_best.n_feat, 'generation': gen + 1,
            'features': [feature_names[i] for i, v in enumerate(gen_best.features) if v],
            'hp': gen_best.hp,
        }

    gen_dur = time.time() - gen_start
    elapsed_min = (time.time() - session_start) / 60
    gens_this_session += 1
    rate = gens_this_session / max(elapsed_min, 0.1) * 60
    remaining_h = SESSION_LIMIT_HOURS - elapsed_min / 60
    gens_remaining_est = int(remaining_h * rate)

    marker = ' *** NEW BEST ***' if improved else ''
    print(f'Gen {gen+1}/{TOTAL_GENS}: best={gen_best.brier:.5f} ({gen_best.model_type}, {gen_best.n_feat}f) '
          f'| ever={best_ever_brier:.5f} | {n_eval} evals {gen_dur:.0f}s '
          f'| {elapsed_min:.0f}min {rate:.0f}g/h ~{gens_remaining_est}g left{marker}')

    if (gen + 1) % 10 == 0:
        mt = Counter(ind.model_type for ind in population)
        top5 = [(f'{ind.brier:.5f}', ind.model_type, ind.n_feat) for ind in population[:5]]
        print(f'  Models: {dict(mt)} | Top5: {top5}')

    save_state(gen, population, best_ever_brier, best_ever_info, mut_rate)

    elite = population[:ELITE_SIZE]

    def tournament(pop, k=4):
        return min(random.sample(pop, min(k, len(pop))), key=lambda x: x.brier)

    children = []
    for e in elite:
        c = Individual(features=list(e.features), model_type=e.model_type, hp=dict(e.hp))
        c.brier = e.brier
        children.append(c)

    while len(children) < POP_SIZE:
        if random.random() < CROSSOVER_RATE:
            child = Individual.crossover(tournament(population), tournament(population))
        else:
            p = tournament(population)
            child = Individual(features=list(p.features), model_type=p.model_type, hp=dict(p.hp))
        child.mutate(mut_rate)
        if random.random() < 0.40:
            child.model_type = 'tabicl'
        children.append(child)

    population = children[:POP_SIZE]
    mut_rate = max(MUT_FLOOR, min(0.15, mut_rate * MUT_DECAY))

save_state(TOTAL_GENS - 1, population, best_ever_brier, best_ever_info, mut_rate)
print(f'\nSession complete: {gens_this_session} gens in {(time.time()-session_start)/60:.0f} min')
print(f'State saved to {STATE_FILE}')
print('To persist state across Kaggle sessions: download evolution_state_kaggle.json')


In [ ]:
# ============================================================
# CELL 6: RESULTS + INJECT BACK INTO HF ISLANDS
# NOTE: Island injection requires internet.
# Switch back to internet-ON mode (turn off GPU), then run this cell.
# ============================================================
import time, json
from pathlib import Path
from collections import Counter

CACHE_DIR = '/kaggle/working'

print('\n' + '='*70)
print('  FINAL RESULTS')
print('='*70)

if best_ever_info:
    print(f'  Best Brier:    {best_ever_info["brier"]:.5f}')
    print(f'  Model:         {best_ever_info["model_type"]}')
    print(f'  Features:      {best_ever_info["n_features"]}')
    print(f'  Generation:    {best_ever_info["generation"]}')
    print(f'  Session gens:  {gens_this_session}')
    print(f'  Platform:      {PLATFORM}')

    population.sort(key=lambda x: x.brier)
    print(f'\n  Top 10:')
    for i, ind in enumerate(population[:10]):
        print(f'    #{i+1}: brier={ind.brier:.5f} | {ind.model_type} | {ind.n_feat}f')

    mt_top = Counter(ind.model_type for ind in population[:20])
    print(f'\n  Models in top 20: {dict(mt_top)}')

    print(f'\n  Current ATR:  0.21837 (S13 CatBoost)')
    delta = best_ever_info['brier'] - 0.21837
    record_str = '*** NEW RECORD! ***' if delta < 0 else ''
    print(f'  Delta vs ATR: {delta:+.5f} {record_str}')

    best_features_file = Path(f'{CACHE_DIR}/best_gpu_features_kaggle.json')
    best_features_file.write_text(json.dumps({
        'brier': best_ever_info['brier'],
        'model_type': best_ever_info['model_type'],
        'features': best_ever_info['features'],
        'n_features': best_ever_info['n_features'],
        'generation': best_ever_info['generation'],
        'source': f'gpu_v2_{PLATFORM}',
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }, indent=2))
    print(f'\n  Best features saved to: {best_features_file}')

    # Inject best individual back into S10
    # Requires internet: switch back to internet-ON mode before running this
    print('\n  Attempting to inject into S10 (requires internet)...')
    try:
        import requests, os
        hf_token = os.environ.get('HF_TOKEN', '')
        if not hf_token:
            from kaggle_secrets import UserSecretsClient
            hf_token = UserSecretsClient().get_secret('HF_TOKEN')

        payload = {
            'inject_individual': {
                'features': best_ever_info['features'],
                'model_type': best_ever_info['model_type'],
                'brier': best_ever_info['brier'],
                'source': f'kaggle_gpu_{PLATFORM}',
            }
        }
        resp = requests.post(
            'https://nomos42-nba-quant.hf.space/api/config',
            json=payload,
            headers={'Authorization': f'Bearer {hf_token}'},
            timeout=15
        )
        if resp.status_code == 200:
            print('  S10 injection: OK')
        else:
            print(f'  S10 injection: {resp.status_code} — {resp.text[:100]}')
    except Exception as e:
        print(f'  S10 injection: failed ({e})')
        print('  -> Use best_gpu_features_kaggle.json to seed islands manually')

    print('\n  -> To inject into other islands: POST /api/config on S11-S15')
    print(f'  -> Best features file: {best_features_file}')

else:
    print('  No results yet — run Cell 5 first.')
